# Notes EA SPORTS FC et valeur marchande

La note FIFA (EA SPORTS FC) et la valeur marchande Transfermarkt sont deux évaluations du même
joueur, faites par des acteurs différents : la première mesure le **niveau actuel** du joueur
dans le jeu, la seconde le **prix** auquel il pourrait être transféré. Elles ne dépendent pas
l'une de l'autre : on cherche ce qu'elles récompensent en commun, et surtout **où et pourquoi
elles divergent**.

**Sommaire**

- Partie A : les notes FIFA
  - [A0. Préparation des données](#A0.-Préparation-des-données)
  - [A1. Distribution des notes](#A1.-Distribution-des-notes)
  - [A2. Note, âge, poste et championnat](#A2.-Note,-âge,-poste-et-championnat)
  - [A3. Note et performances réelles](#A3.-Note-et-performances-réelles)
- Partie B : comparaison avec la valeur marchande
  - [B1. Le lien entre note et valeur](#B1.-Le-lien-entre-note-et-valeur)
  - [B2. Mêmes facteurs, poids différents](#B2.-Mêmes-facteurs,-poids-différents)
  - [B3. Écart entre le marché et FIFA](#B3.-Écart-entre-le-marché-et-FIFA)
  - [B4. Évolution sur quatre saisons](#B4.-Évolution-sur-quatre-saisons)

# Partie A : les notes FIFA

## A0. Préparation des données

On reprend les joueurs de l'analyse des valeurs marchandes (au moins 900 minutes jouées en
2025-26) et on leur associe leur note dans **EA SPORTS FC 27**. Cette édition est sortie en
septembre 2026 : sa note reflète la saison 2025-26, comme la valeur au 30 juin 2026.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

from valeur_marchande.transfermarkt import load_snapshot, load_panel
from valeur_marchande.ea_fc import load_matched, add_ratings_to_panel
from valeur_marchande.analyse import (ORDRE_POSTES, FACTEURS, preparer_joueurs,
                                      tableau_coefficients, importance_facteurs)

In [2]:
joueurs = preparer_joueurs(load_snapshot())
notes = load_matched()   # une ligne par joueur et par édition (FIFA 23 à FC 27)

notes["edition"].value_counts()

edition
fc24      6385
fc27      6310
fc26      6118
fifa23    6117
fc25      5793
Name: count, dtype: int64

In [3]:
COLONNES_NOTES = ["player_id", "overall_rating", "pace", "shooting", "passing",
                  "dribbling", "defending", "physical"]
notes_fc27 = notes.loc[notes["edition"] == "fc27", COLONNES_NOTES]
joueurs_notes = joueurs.merge(notes_fc27, on="player_id")
joueurs_notes = joueurs_notes.rename(columns={"overall_rating": "note"})

print(f"{len(joueurs_notes)} joueurs avec une note FC 27, sur {len(joueurs)} "
      f"({len(joueurs_notes) / len(joueurs):.0%})")

2678 joueurs avec une note FC 27, sur 2942 (91%)


Les joueurs sans note sont ceux que le rapprochement entre Transfermarkt et EA n'a pas
retrouvés (nom ou date de naissance trop différents, joueur absent du jeu). Sont-ils différents
des autres ?

In [4]:
joueurs["a_une_note"] = joueurs["player_id"].isin(notes_fc27["player_id"])
joueurs.groupby("a_une_note")[["valeur_millions", "age", "minutes_2025_26"]].median()

,valeur_millions,age,minutes_2025_26
a_une_note,,,
False,0.9,29.0,1527.0
True,6.0,26.0,2169.5


91 % des joueurs de l'analyse (2 678 sur 2 942) ont une note FC 27. Les joueurs sans note sont vraiment moins valorisés (valeur médiane de 0,9 M€ contre 6,0 M€), un peu plus âgés (29 ans contre 26) et jouent moins (1 527 minutes contre 2 170). Il s'agit surtout de joueurs moins connus, ou absents du jeu. L'échantillon étudié est donc orienté vers les bons joueurs : c'est une limite à garder en tête, les résultats suivants portent surtout sur les joueurs les mieux valorisés.